In [8]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

ds_name = "cifar"
split = "trainUval"

epoch = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
dses = find_all_datasets("../../datasets/")
ds = dses[ds_name]

In [10]:
data_dir = "C:/home/ae_data/landscape_data"
strees_dir = "C:/home/ae_data/strees_ae/"

In [11]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[0]
for e in exps:
	if e.dataset.name == ds_name and e.split == split and e.epoch == epoch:
		exp = e
		break

assert exp.dataset.name == ds_name
assert exp.split == split
assert exp.epoch == epoch

e

cifar (cifar-trainUval), k=20, layer=latents, epoch=100

In [12]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [13]:
import pyct as ct

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

homo_prop = 0.99
fns, counts = simpl.getHomoValleyPlot(order, wts, labels, homo_prop, partition) # type: ignore
fns_norm, counts_norm_min, counts_norm_max = simpl.getSimplificationPlot(order, wts)

In [14]:
def count_stability(valleys, fns):    
	stabilities = {}
	
	for i, fn in enumerate(fns[1:]):
		i += 1
		
		val = valleys[i]
		if val < valleys[i - 1]:
			stabilities[valleys[i - 1]] = (fn - fns[i - 1], fns[i - 1])

	stabilities[valleys[-1]] = (fns[-1], fns[-1])
	
	return stabilities
		

In [15]:
import plotly.express as px

px.line(x=fns, y=counts, labels={"x": "Function Value", "y": "Number of Homogeneous Valleys"}, title=f"Homogeneous Valleys vs Function Value (Homogeneity Threshold = {homo_prop})")

In [16]:
import pandas as pd

stab_df = pd.DataFrame([(k, v, last_value) for k, (v, last_value) in count_stability(counts, fns).items()], columns=["hvalleys", "stability", "fn_end"])
stab_df.sort_values(by=["stability"], ascending=False).head(10)

,hvalleys,stability,fn_end
3902,0,0.308950,0.308950
3899,3,0.014217,0.133356
3901,1,0.010958,0.163004
3900,2,0.002546,0.149301
3897,5,0.001741,0.112604
3893,9,0.001492,0.105040
3874,28,0.001450,0.085109
3890,12,0.001213,0.097627
3898,4,0.001072,0.114345
3869,33,0.000922,0.081045


In [17]:
px.line(x=fns_norm, y=counts_norm_min, labels={"x": "Function Value", "y": "Number of Valleys"}, title="Number of Valleys vs Function Value")

In [18]:
import pandas as pd

stab_df = pd.DataFrame([(k, v, last_value) for k, (v, last_value) in count_stability(counts_norm_min, fns_norm).items()], columns=["valleys", "stability", "fn_end"])
stab_df.sort_values(by=["stability"], ascending=False).head(10)

,valleys,stability,fn_end
5231,1,0.308950,0.308950
5230,2,0.134988,0.173963
5224,8,0.014217,0.133356
5228,4,0.011157,0.151847
5229,3,0.010958,0.163004
5223,9,0.007175,0.126180
5212,20,0.004987,0.106768
5219,13,0.004771,0.117078
5200,32,0.002816,0.094811
5227,5,0.002546,0.149301


In [19]:
data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

fns_more, remaining_all, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = simpl.getHomoValleyPlotPlusCoverages(order, wts, labels, homo_prop, partition)

In [20]:
classes_needed = 9 # 10 classes, the following criteria must apply to this many of them
maj_coverage_needed = 0.01 # at least 1% coverage in homogeneous valleys where they are the only class (100% proportion in valley) 
total_coverage_needed = 0.02 # at least 5% total coverage in all valleys

# coverages are functions of accuracy, so this constraint necessarily tightens in more complex datasets

represented = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) >= classes_needed 
            	for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]

# find interval where this is true
last_idx = len(fns) - represented[::-1].index(True) - 1

first_idx = represented.index(True)

print(f"Uniform Interval: {all(represented[first_idx:last_idx+1])}")
print(f"First IDX: {first_idx} - ")
print(first_idx, len(fns), fns[first_idx], list(zip(maj_class_homo_cov[first_idx], class_all_covs[first_idx], maj_class_homo_counts[first_idx])), sep="\n")
print(f"\nLast IDX: {last_idx} - ")
print(last_idx, len(fns), fns[last_idx], list(zip(maj_class_homo_cov[last_idx], class_all_covs[last_idx], maj_class_homo_counts[last_idx])), sep="\n")

Uniform Interval: True
First IDX: 0 - 
0
5232
0.0
[(0.054, 0.08633333333333333, 278), (0.101, 0.1495, 525), (0.052, 0.09633333333333334, 282), (0.0965, 0.16433333333333333, 513), (0.033166666666666664, 0.07333333333333333, 172), (0.117, 0.17866666666666667, 613), (0.030666666666666665, 0.068, 160), (0.09833333333333333, 0.14866666666666667, 499), (0.06383333333333334, 0.10083333333333333, 323), (0.10566666666666667, 0.1515, 547)]

Last IDX: 3528 - 
3528
5232
0.02432835102081299
[(0.018833333333333334, 0.035166666666666666, 92), (0.029666666666666668, 0.048, 142), (0.014, 0.0315, 75), (0.035333333333333335, 0.06883333333333333, 181), (0.008666666666666666, 0.022333333333333334, 43), (0.04466666666666667, 0.07466666666666667, 232), (0.01, 0.021666666666666667, 50), (0.027, 0.048, 131), (0.021333333333333333, 0.035333333333333335, 103), (0.03183333333333333, 0.049666666666666665, 157)]
